In [3]:
import os
import sys
import numpy as np
import schedule
import time
import astropy.units as u
import astropy.time
from astropy.coordinates import EarthLocation

In [ ]:
def get_utc_from_lst(target_lst_hrs, date_str, longitude_deg=-118.3):

    loc = EarthLocation.from_geodetic(lon=longitude_deg * u.deg, lat=0 * u.deg)
    reference_time = astropy.time.Time(f"{date_str[:4]}-{date_str[4:6]}-{date_str[6:8]} 12:00:00", scale='utc', location=loc)
    reference_lst = reference_time.sidereal_time('mean').hour
    lst_diff = (target_lst_hrs - reference_lst + 12) % 24 - 12
    utc = reference_time + (lst_diff * 0.99726958) * u.hour
    if isinstance(utc.datetime, (list, np.ndarray)):
        array_shape = np.shape(utc)
        utc_str = np.array([f"{use_utc.datetime.hour:02d}{use_utc.datetime.minute:02d}{use_utc.datetime.second:02d}" for use_utc in utc.flatten()])
        utc_str = np.reshape(utc_str, shape=array_shape)
    else:
        utc_str = f"{utc.datetime.hour:02d}{utc.datetime.minute:02d}{utc.datetime.second:02d}"
    
    return utc_str

In [22]:
lst_ranges=[
    [11.88, 14.88],  # 3 hours of cosmology data
    [17.38, 17.88],  # half an hour of calibration data
]
print(get_utc_from_lst(lst_ranges, "20260409", longitude_deg=-118.3))

(2, 2)
[['063553' '093524']
 ['120459' '123454']]


In [11]:
utc = get_utc_from_lst(11+53/60.0, "20260409", longitude_deg=-118.3)
print(utc)

063605


In [12]:
utc = get_utc_from_lst(11+53/60.0, "20260401", longitude_deg=-118.3)
print(utc)

070733


In [20]:
utc = get_utc_from_lst(np.array([[11+53/60.0, 14+53/60.],[11+53/60.0, 14+53/60.]]), "20260409", longitude_deg=-118.3)

(2, 2)


In [21]:
utc

array([['063605', '093536'],
       ['063605', '093536']], dtype='<U6')

In [12]:
utc.value

array(['2026-04-09 06:36:05.855', '2026-04-09 09:35:36.366'], dtype='<U23')

In [ ]:

print(f"{utc.datetime.hour:02d}{utc.datetime.minute:02d}{utc.datetime.second:02d}")

6
36
5
063605


In [12]:
target_dir="/lustre/pipeline/cosmology"
source_dir="/lustre/pipeline/slow"
copy_freqs=[
    "27",
    "32",
    "36",
    "41",
    "46",
    "50",
    "55",
    "59",
    "64",
    "69",
    "73",
    "78",
    "82",
]
copy_freqs=["46"]
time_ranges=[
    ["123000", "130000"],  # half an hour of calibration data
    ["070000", "100000"],  # three hours of cosmology data
]
use_dates=None

for freq in copy_freqs:
    dates = np.sort(os.listdir(f"{source_dir}/{freq}MHz"))
    if use_dates is not None:
        dates = [date for date in dates if date in use_dates]
    for date in dates:
        hours = np.sort(os.listdir(f"{source_dir}/{freq}MHz/{date}"))
        for hour in hours:
            filenames = np.sort(
                os.listdir(f"{source_dir}/{freq}MHz/{date}/{hour}")
            )
            for filename in filenames:
                timestamp = filename[9:15]
                copy_file = False
                for time_range in time_ranges:
                    if int(time_range[0]) < int(timestamp) < int(time_range[1]):
                        copy_file = True
                        break
                if copy_file:
                    filename = f"{filename[:16]}{freq}{filename[18:]}"
                    if os.path.isdir(
                        f"{source_dir}/{freq}MHz/{date}/{hour}/{filename}"
                    ):
                        # copy_command = f"sudo mkdir -p {target_dir}/{freq}MHz/{date}/{hour} && sudo cp -r {source_dir}/{freq}MHz/{date}/{hour}/{filename} {target_dir}/{freq}MHz/{date}/{hour}"
                        copy_command = f"mkdir -p {target_dir}/{freq}MHz/{date}/{hour} && sudo mv {source_dir}/{freq}MHz/{date}/{hour}/{filename} {target_dir}/{freq}MHz/{date}/{hour}"
                        print(f"Copying {filename}")
                        os.system(copy_command)
                    else:
                        continue

Copying 20260407_074918_46MHz.ms
Copying 20260407_074928_46MHz.ms
Copying 20260407_074938_46MHz.ms
Copying 20260407_074948_46MHz.ms
Copying 20260407_074958_46MHz.ms
Copying 20260407_075008_46MHz.ms
Copying 20260407_075018_46MHz.ms
Copying 20260407_075028_46MHz.ms
Copying 20260407_075038_46MHz.ms
